In [ ]:
import os
import numpy as np
import pandas as pd
import seaborn as sns

import glob
import matplotlib.pyplot as plt
import plotly
import plotly.express as px
import plotly.graph_objs as go

# from Bio import SeqIO

import gzip
import h5py
import scanpy as sc
import scipy
import mira
import torch

In [ ]:
mira.__version__,torch.__version__

In [ ]:
import logging
import warnings
mira.utils.pretty_sderr()

In [ ]:
%matplotlib inline

In [ ]:
torch.cuda.is_available()

In [ ]:
for i in range(torch.cuda.device_count()):
    print(torch.cuda.get_device_properties(i).name)

In [ ]:
torch.cuda.get_device_properties(0)

In [ ]:
cd /media/RAIDArray/Nick/20240206_PerturbSeq_Exp2/cellranger_output

In [ ]:
pwd

# merge samples

In [ ]:
filenames = glob.glob("/media/RAIDArray/Nick/20240206_PerturbSeq_Exp2/cellranger_output/*/sample_filtered_feature_bc_matrix.h5")

adatas=[]
for filename in filenames:
    adatas.append(sc.read_10x_h5(filename))
    adatas[-1].var_names_make_unique()
    adatas[-1].obs['group']=filename.split('/')[-2]
    

adata = adatas[0].concatenate(adatas[1:],index_unique=None)

adata

In [ ]:
adata.obs['group'].unique()

In [ ]:
adata.obs.group.unique()

In [ ]:
import pandas as pd
import anndata as ad

# Assuming 'adata' is your AnnData object

def remove_var_columns(adata: ad.AnnData, columns_to_remove: list[str]):
  """
  Removes specified columns from the .var DataFrame of an AnnData object.

  Args:
      adata: The AnnData object.
      columns_to_remove: A list of column names to remove.
  """

  var_columns = adata.var.columns
  columns_to_keep = [col for col in var_columns if col not in columns_to_remove]
  adata.var = adata.var[columns_to_keep]


# Example usage:
columns_to_remove = ['pattern', 'read', 'sequence']  # List of columns to remove
remove_var_columns(adata, columns_to_remove)

print(adata.var.columns)  # Print the remaining columns to verify

In [ ]:
adata.var

In [ ]:
adata.write("/media/RAIDArray/Nick/20240206_PerturbSeq_Exp2/cellranger_output/tf_perturbseq_rep2_post_cellranger.h5ad")

# SCANPY preprocessing

In [ ]:
adata = sc.read_h5ad("/media/RAIDArray/Nick/20240206_PerturbSeq_Exp2/cellranger_output/tf_perturbseq_rep2_post_cellranger.h5ad")
adata

In [ ]:
sc.pp.filter_cells(adata, min_genes=200)
sc.pp.filter_genes(adata, min_cells=20)

In [ ]:
adata.var['mt'] = adata.var_names.str.startswith('MT-')

In [ ]:
adata

In [ ]:
sc.pp.calculate_qc_metrics(adata, qc_vars=['mt'], percent_top=None, log1p=False, inplace=True)

In [ ]:
sc.pl.violin(adata, ['n_genes_by_counts', 'total_counts', 'pct_counts_mt'],
             jitter=0.4,
             multi_panel=True
            )

In [ ]:
sc.pl.scatter(adata, x='total_counts', y='pct_counts_mt')
sc.pl.scatter(adata, x='total_counts', y='n_genes_by_counts')

In [ ]:
adata = adata[adata.obs.n_genes_by_counts < 7000, :]
adata = adata[adata.obs.pct_counts_mt < 5, :]

In [ ]:
adata

In [ ]:
adata.write('/ix/djishnu/peasena/tf_perturbseq/20240206_perturbseq2/merged.h5ad')

# MIRA

In [ ]:
adata=sc.read_h5ad("/ix/djishnu/peasena/tf_perturbseq/20240206_perturbseq2/merged.h5ad")

In [ ]:
adata.var['gene_ids'].head(200)

In [ ]:
adata.obs_names_make_unique()

In [ ]:
rawdata = adata.X.copy()

In [ ]:
sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)

In [ ]:
adata.layers['counts'] = rawdata

In [ ]:
sc.pp.highly_variable_genes(adata, min_disp = 0.2)

In [ ]:
# sc.pp.highly_variable_genes(adata, min_mean=0.0125, max_mean=3, min_disp=0.5)
# sc.pp.highly_variable_genes(adata, min_disp = 0.2)

In [ ]:
sc.pl.highly_variable_genes(adata)

In [ ]:
adata

In [ ]:
adata.var['highly_variable'].value_counts()

In [ ]:
sc.tl.pca(adata)
sc.pp.neighbors(adata, n_pcs=50)
sc.tl.umap(adata, min_dist = 0.2, negative_sample_rate=0.2)
sc.pl.umap(adata, color = 'group', frameon=False)

In [ ]:
model = mira.topics.make_model(
    adata.n_obs, adata.n_vars, # helps MIRA choose reasonable values for some hyperparameters which are not tuned.
    feature_type = 'expression',
    highly_variable_key='highly_variable',
    counts_layer='counts',
#     categorical_covariates='batch'
)

In [ ]:
model.get_learning_rate_bounds(adata)

In [ ]:
model.set_learning_rates(1e-3, 0.25)
model.plot_learning_rate_bounds(figsize=(7,3))

# Hyperparameter Optimization: Gradient based

In [ ]:
#takes a long time (30min)
topic_contributions = mira.topics.gradient_tune(model, adata)

In [ ]:
with open('/ix/djishnu/peasena/tf_perturbseq/20240206_perturbseq2/topic_contributions.txt', 'w') as file:
    file.write('\n'.join(str(topic) for topic in topic_contributions))

In [ ]:
NUM_TOPICS = 33

mira.pl.plot_topic_contributions(topic_contributions, NUM_TOPICS)

In [ ]:
#takes ~5-10min
model = model.set_params(num_topics = NUM_TOPICS).fit(adata)

In [ ]:
#reload model if needed
model = mira.topic_model.load_model('/ix/djishnu/peasena/tf_perturbseq/20240206_perturbseq2/models/expression_model_tuner/base.pth')

In [ ]:
model

# Hyperparameter Optimization: Bayesian

In [ ]:
#reload adata if needed
adata=sc.read_h5ad("/ix/djishnu/peasena/tf_perturbseq/20240206_perturbseq2/merged_gradient_topicmodel.h5ad")

In [ ]:
tuner = mira.topics.BayesianTuner(
        model = model,
        n_jobs=2,
        save_name = '/ix/djishnu/peasena/tf_perturbseq/20240206_perturbseq2/results/expression_model_tuner',
        #### IMPORTANT
        min_topics = 35, max_topics = 37, # tailor for your dataset!!!! (+/- 10 from the above topic number)
        #### See "Notes on min_topics, max_topics" above
        #storage = mira.topics.Redis() # if using REDIS backend for more (>5) processes
)

In [ ]:
#takes the longest (~2-4hrs)
tuner.fit(adata)

In [ ]:
ax = tuner.plot_intermediate_values(palette='Spectral_r',
                                   log_hue=True, figsize=(7,3))
# ax.set(ylim = (7e2, 7.7e2))

In [ ]:
tuner.plot_pareto_front(include_pruned_trials=False, label_pareto_front=True,
                       figsize = (5,5))

In [ ]:
model = tuner.fetch_best_weights()

In [ ]:
model.save('/ix/djishnu/peasena/tf_perturbseq/20240206_perturbseq2/results/mira_model_240401.pth')

In [ ]:
#uses topic model to predict for each cell
model.predict(adata)


In [ ]:
#reload model if needed
model = mira.topic_model.load_model('/ix/djishnu/peasena/tf_perturbseq/20240206_perturbseq2/results/mira_model_240401.pth')

In [ ]:
adata

In [ ]:
# add day
cond = []
for x in adata.obs.group:
    x = x[0:2]
    if x == 'D4':
        cond.append('D4')
    elif x == 'D6':
        cond.append('D6')

adata.obs['day'] = cond

In [ ]:
# add knockout
cond = []
for x in adata.obs.group:
    x = x.split('_')[1]
    if x == 'batf':
        cond.append('BATF')
    elif x == 'irf4':
        cond.append('IRF4')
    elif x == 'irf8':
        cond.append('IRF8')
    elif x == 'ntc':
        cond.append('Control')
    elif x == 'prdm1':
        cond.append('PRDM1')
    elif x == 'spib':
        cond.append('SPIB')

adata.obs['knockout'] = cond

In [ ]:
# add replicate
cond = []
for x in adata.obs.group:
    x = x.split('_')[2]
    if x == 'r1':
        cond.append('Rep1')
    elif x == 'r2':
        cond.append('Rep2')

adata.obs['replicate'] = cond

In [ ]:
#add sample column
for x in adata.obs.group:
    adata.obs[x] = np.where(adata.obs.group == x, "1", "0")

In [ ]:
#add colors
adata.uns['D4_ntc_r2_colors'] =  ['darkgrey','red']
adata.uns['D6_ntc_r2_colors'] =  ['darkgrey','red']


In [ ]:
sc.set_figure_params(scanpy=True, fontsize=10, dpi_save = 350)

plt.rcParams["savefig.dpi"] = 350

In [ ]:
model.get_umap_features(adata, box_cox=0.1)
sc.pp.neighbors(adata, use_rep = 'X_umap_features', metric = 'manhattan',n_neighbors=20)
sc.tl.umap(adata, min_dist=0.25, negative_sample_rate=5,random_state=0)

In [ ]:
adata.write('/ix/djishnu/peasena/tf_perturbseq/20240206_perturbseq2/results/240401_merged.h5ad')